## 0. Введение

Этот ноутбук демонстрирует базовый запуск обучения распознавателя `TRBA` в версии `0.1.10`. В примере для быстрого теста используется валидационная часть набора данных как обучающая и валидационная выборка, поэтому он предназначен прежде всего для демонстрации процесса запуска обучения.

Пример был апробирован в среде Google Colab в конфигурации с 12 ГБ оперативной памяти и видеокартой NVIDIA T4.

Минимальные технические требования для запуска примера:

- Python 3.8+
- не менее 12 ГБ оперативной памяти
- NVIDIA GPU с объемом видеопамяти не менее 8 ГБ

Следует учитывать, что данный пример относится к обучению модели распознавания, поэтому запуск на GPU является рекомендуемым сценарием. Запуск на CPU возможен, но будет существенно медленнее и не рассматривается как базовый вариант для апробации.

## 1. Установка зависимостей

In [ ]:
#может потребовать перезапуска среды в Google Colab
!pip install manuscript-ocr[dev]==0.1.10

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 959.8/959.8 kB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 105.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.3/69.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 106.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 98.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.

In [ ]:
# Затем обновите PyTorch на GPU версию, совместимую с вашей версией CUDA (например, для CUDA 11.8):
pip install --upgrade torch torchvision --index-url https://download.pytorch.org/whl/cu118

In [1]:
!pip install rarfile

## 2. Скачивание данных

In [3]:
from pathlib import Path
from urllib.request import urlretrieve

import rarfile

url = "https://huggingface.co/datasets/sherstpasha/YeniseiGovReports-HWR/resolve/main/YeniseiGovReports-HWR.rar"

work_dir = Path("datasets")
archive_path = work_dir / "YeniseiGovReports-HWR.rar"
extract_dir = work_dir / "YeniseiGovReports-HWR"

work_dir.mkdir(parents=True, exist_ok=True)

if not archive_path.exists():
    print("Downloading dataset...")
    urlretrieve(url, archive_path)
    print(f"Downloaded to: {archive_path}")

if not extract_dir.exists():
    extract_dir.mkdir(parents=True, exist_ok=True)

if not any(extract_dir.iterdir()):
    print("Extracting archive...")
    try:
        with rarfile.RarFile(archive_path) as rf:
            rf.extractall(extract_dir)
    except rarfile.RarCannotExec as e:
        raise RuntimeError(
            "rarfile установлен, но в системе не найден архиватор для распаковки RAR. "
            "Установите один из архиваторов: unrar, unar, 7z или 7zz."
        ) from e
    print(f"Extracted to: {extract_dir}")

print("archive_path =", archive_path)
print("extract_dir  =", extract_dir)


Downloaded to: datasets/YeniseiGovReports-HWR.rar
Extracting archive...
Extracted to: datasets/YeniseiGovReports-HWR
archive_path = datasets/YeniseiGovReports-HWR.rar
extract_dir  = datasets/YeniseiGovReports-HWR


## 3. Запуск обучения

In [4]:
from pathlib import Path

from manuscript.recognizers import TRBA

dataset_root = Path(extract_dir) / "YeniseiGovReports-HWR" / "val"

val_csvs = str(dataset_root / "labels.csv")
val_roots = str(dataset_root / "img")

# Для быстрого теста используем валидационный набор и как train, и как val
train_csvs = val_csvs
train_roots = val_roots

print("train_csvs  =", train_csvs)
print("train_roots =", train_roots)
print("val_csvs    =", val_csvs)
print("val_roots   =", val_roots)

TRBA.train(
    train_csvs=train_csvs,
    val_csvs=val_csvs,
    train_roots=train_roots,
    val_roots=val_roots,
    cnn_backbone="seresnet31lite",
    epochs=1,
)

[2026-04-08 12:16:34] Start training
INFO:train:Start training
[2026-04-08 12:16:34] Experiment dir: exp1
INFO:train:Experiment dir: exp1
[2026-04-08 12:16:34] Seed: 42
INFO:train:Seed: 42
[2026-04-08 12:16:34] Saved config to exp_dir/config.json
INFO:train:Saved config to exp_dir/config.json
[2026-04-08 12:16:34] Device: cuda
INFO:train:Device: cuda
[2026-04-08 12:16:34] Charset loaded: 194 tokens
INFO:train:Charset loaded: 194 tokens
[2026-04-08 12:16:34] Charset copied to experiment dir: exp1/charset.txt
INFO:train:Charset copied to experiment dir: exp1/charset.txt


train_csvs  = datasets/YeniseiGovReports-HWR/YeniseiGovReports-HWR/val/labels.csv
train_roots = datasets/YeniseiGovReports-HWR/YeniseiGovReports-HWR/val/img
val_csvs    = datasets/YeniseiGovReports-HWR/YeniseiGovReports-HWR/val/labels.csv
val_roots   = datasets/YeniseiGovReports-HWR/YeniseiGovReports-HWR/val/img


[2026-04-08 12:16:35] Using default pretrained weights: trba_lite_g1.pth (GitHub release)
INFO:train:Using default pretrained weights: trba_lite_g1.pth (GitHub release)
[2026-04-08 12:16:35] Default pretrain config: https://github.com/konstantinkozhin/manuscript-ocr/releases/download/v0.1.0/trba_lite_g1.json
INFO:train:Default pretrain config: https://github.com/konstantinkozhin/manuscript-ocr/releases/download/v0.1.0/trba_lite_g1.json


Downloading: "https://github.com/konstantinkozhin/manuscript-ocr/releases/download/v0.1.0/trba_lite_g1.pth" to /root/.cache/torch/hub/checkpoints/trba_lite_g1.pth


100%|██████████| 36.2M/36.2M [00:03<00:00, 10.9MB/s]
[2026-04-08 12:16:39] Pretrain load summary: loaded=245/245 keys; missing=0; shape_mismatch=0
INFO:train:Pretrain load summary: loaded=245/245 keys; missing=0; shape_mismatch=0
[2026-04-08 12:16:39] Freeze policy applied: cnn: NONE (no freezing)
INFO:train:Freeze policy applied: cnn: NONE (no freezing)
[2026-04-08 12:16:39] Freeze policy applied: enc_rnn: NONE (no freezing)
INFO:train:Freeze policy applied: enc_rnn: NONE (no freezing)
[2026-04-08 12:16:39] Freeze policy applied: attention: NONE (no freezing)
INFO:train:Freeze policy applied: attention: NONE (no freezing)
[2026-04-08 12:16:39] Parameters: trainable=10,618,692 | frozen=0 | total=10,618,692
INFO:train:Parameters: trainable=10,618,692 | frozen=0 | total=10,618,692
/usr/local/lib/python3.12/dist-packages/manuscript/recognizers/_trba/training/train.py:825: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` 

...

Train 1/1:   0%|          | 0/700 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/manuscript/recognizers/_trba/training/train.py:1081: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():
Train 1/1:   0%|          | 0/700 [00:17<?, ?it/s, loss=4.8560, lr=8.00e-05]/usr/local/lib/python3.12/dist-packages/manuscript/recognizers/_trba/training/train.py:1138: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
Valid Set 0 1/1:   0%|          | 0/700 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/manuscript/recognizers/_trba/training

[2026-04-08 12:22:10] Epoch 001/1 | train_loss=0.5045 | val_loss=0.1069 | acc=0.8804 | CER=0.0336 | WER=0.1250 | lr=2.04e-08
INFO:train:Epoch 001/1 | train_loss=0.5045 | val_loss=0.1069 | acc=0.8804 | CER=0.0336 | WER=0.1250 | lr=2.04e-08


Epoch 001/1 | train_loss=0.5045 | val_loss=0.1069 | acc=0.8804 | CER=0.0336 | WER=0.1250 | lr=2.04e-08


[2026-04-08 12:22:15] New best val_loss: 0.1069 (epoch 1)
INFO:train:New best val_loss: 0.1069 (epoch 1)
[2026-04-08 12:22:16] New best acc: 0.8804 (epoch 1)
INFO:train:New best acc: 0.8804 (epoch 1)
[2026-04-08 12:22:16] Training finished.
INFO:train:Training finished.
[2026-04-08 12:22:16] Attempting to export best model to ONNX...
INFO:train:Attempting to export best model to ONNX...
[2026-04-08 12:22:16] Failed to export ONNX model: cannot import name 'export_to_onnx' from 'manuscript.recognizers._trba' (/usr/local/lib/python3.12/dist-packages/manuscript/recognizers/_trba/__init__.py)


{'val_acc': 0.8804464285714285,
 'val_loss': 0.1069488455894004,
 'exp_dir': 'exp1'}

## 4. Экспорт модели в формат ONNX

In [7]:
!pip install onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.1/164.1 kB 15.2 MB/s eta 0:00:00


In [8]:
from pathlib import Path

from manuscript.recognizers import TRBA

exp_dir = Path("exp1")

TRBA.export(
    weights_path=str(exp_dir / "best_acc_weights.pth"),
    config_path=str(exp_dir / "config.json"),
    charset_path=str(exp_dir / "charset.txt"),
    output_path=str(exp_dir / "trba_onnx_model.onnx"),
)

Loading config from exp1/config.json...
Loading charset from exp1/charset.txt...
Charset loaded: 194 total classes (including special tokens)
  First 4 tokens (special): ['<PAD>', '<SOS>', '<EOS>', ' ']
  Regular characters: 190

Loading checkpoint from exp1/best_acc_weights.pth...

=== TRBA ONNX Export ===
Max decoding length: 25
Input size: 64x256
Architecture: seresnet31lite
Hidden size: 256
Num classes: 194

Creating model architecture...
   Token IDs:
      SOS:   1
      EOS:   2
      PAD:   0
      BLANK: None
      SPACE: 3
Loading weights into model...
[OK] Model loaded

Creating ONNX wrapper...
   max_length from config: 25
   ONNX will use: 26 steps (max_length + 1 for compatibility)
Input shape: torch.Size([1, 3, 64, 256])

Testing model before export...
Output shape: torch.Size([1, 26, 194])
Expected: [1, 26, 194] (max_length + 1 steps)

Exporting to ONNX (opset 14)...


/usr/local/lib/python3.12/dist-packages/manuscript/recognizers/_trba/__init__.py:1050: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0408 12:26:34.772000 1883 torch/onnx/_internal/exporter/_compat.py:125] Setting ONNX exporter to use operator set version 18 because the requested opset_version 14 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
W0408 12:26:35.555000 1883 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_sc

Applied 111 of general pattern rewrite rules.
[OK] ONNX model saved to: exp1/trba_onnx_model.onnx

Verifying ONNX model...
[OK] ONNX model is valid

Simplifying ONNX model...
[OK] ONNX model simplified

Testing ONNX inference...
[OK] ONNX inference works!
  Output shape: (1, 26, 194)
  Max difference vs PyTorch: 0.000029
  [OK] Outputs match!

[OK] Export complete! Model size: 40.4 MB

Input shape: [batch_size, 3, 64, 256]
Output shape: [batch_size, 25, 194]
Decoding: Greedy (argmax over last dimension)


## 5. Инференс экспортированной модели

После обучения и экспорта модель распознавания можно сразу использовать для инференса. В данном примере загружается тестовый фрагмент `crop1.png` из основного репозитория и выполняется распознавание с помощью экспортированной модели `TRBA`.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

from manuscript.recognizers import TRBA

exp_dir = Path("exp1")

image_url = "https://raw.githubusercontent.com/konstantinkozhin/manuscript-ocr/main/example/images/crop1.png"
image_path = Path("crop1.png")

if not image_path.exists():
    urlretrieve(image_url, image_path)

recognizer = TRBA(
    weights=str(exp_dir / "trba_onnx_model.onnx"),
    config=str(exp_dir / "config.json"),
    charset=str(exp_dir / "charset.txt"),
)

result = recognizer.predict(str(image_path))

print("Результат распознавания:")
print(result)
print("Текст:", result[0]["text"])
print("Confidence:", result[0]["confidence"])

Таким образом, библиотека `manuscript-ocr` предоставляет возможность обучать распознаватели на собственных данных. Более подробная информация о параметрах обучения и доступных настройках приведена в полной документации проекта: https://konstantinkozhin.github.io/manuscript-ocr/0.1.10/en/api/recognizers.html